In [ ]:
!pip install -q torch torchvision torchaudio pytorchvideo \
  opencv-python-headless scikit-learn matplotlib \
  tqdm numpy pandas pillow ultralytics

In [ ]:
import os
import sys
import gc
import time
import random
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from tqdm import tqdm
import pickle
import json
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.9.0+cu126
CUDA available: True
Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class DatasetConfig:
    """Configuration for custom dataset"""

    # Your dataset paths
    INPUT_VIDEO_DIR = '/content/drive/MyDrive/Graduation_project/Training-Normal-Videos-Part-1'

    # Output directory for features
    OUTPUT_BASE_DIR = '/content/drive/MyDrive/Graduation_project'
    FEATURES_DIR_NAME = 'features'  # Folder name to create

    # Feature extraction settings
    FRAME_SIZE = (224, 224)  # Resize frames to this size
    SEGMENT_LENGTH = 16  # Number of frames per segment
    FPS = 30  # Frames per second
    MAX_FRAMES = 3000  # Maximum frames per video (for memory)

    # Motion extractor
    MOTION_EXTRACTOR = 'i3d'  # 'c3d', 'i3d', 'r3d', 'lightweight', or 'yolo'
    FEATURE_DIMS = {
        'c3d': 4096,
        'i3d': 1024,
        'r3d': 512,
        'lightweight': 512,
        'yolo': 83
    }

    # Output directories (will be created automatically)
    FEATURES_DIR = None
    METADATA_DIR = None
    CHECKPOINTS_DIR = None

    @classmethod
    def setup_directories(cls):
        """Create necessary directories"""
        # Create features directory
        cls.FEATURES_DIR = os.path.join(cls.OUTPUT_BASE_DIR, cls.FEATURES_DIR_NAME)
        cls.METADATA_DIR = os.path.join(cls.FEATURES_DIR, 'metadata')
        cls.CHECKPOINTS_DIR = os.path.join(cls.FEATURES_DIR, 'checkpoints')

        dirs = [cls.FEATURES_DIR, cls.METADATA_DIR, cls.CHECKPOINTS_DIR]

        for dir_path in dirs:
            os.makedirs(dir_path, exist_ok=True)
            print(f"Created directory: {dir_path}")

        return dirs

# Setup paths
print("=" * 70)
print("SETTING UP DIRECTORIES")
print("=" * 70)
print(f"Input video directory: {DatasetConfig.INPUT_VIDEO_DIR}")
print(f"Output base directory: {DatasetConfig.OUTPUT_BASE_DIR}")

# Create directories
features_dir, metadata_dir, checkpoints_dir = DatasetConfig.setup_directories()

print(f"\n✅ Directories created successfully!")
print(f"   Features will be saved to: {features_dir}")
print(f"   Metadata will be saved to: {metadata_dir}")

SETTING UP DIRECTORIES
Input video directory: /content/drive/MyDrive/Graduation_project/Training-Normal-Videos-Part-1
Output base directory: /content/drive/MyDrive/Graduation_project
Created directory: /content/drive/MyDrive/Graduation_project/features
Created directory: /content/drive/MyDrive/Graduation_project/features/metadata
Created directory: /content/drive/MyDrive/Graduation_project/features/checkpoints

✅ Directories created successfully!
   Features will be saved to: /content/drive/MyDrive/Graduation_project/features
   Metadata will be saved to: /content/drive/MyDrive/Graduation_project/features/metadata


In [ ]:
class CustomDatasetMetadata:
    """Load and manage custom dataset metadata"""

    @staticmethod
    def get_all_videos(video_dir):
        """Get all videos from dataset directory"""
        videos = []

        if not os.path.exists(video_dir):
            print(f"❌ Video directory not found: {video_dir}")
            return videos

        # Get all video files
        video_extensions = ('.avi', '.mp4', '.mov', '.mkv', '.flv', '.wmv')

        for root, dirs, files in os.walk(video_dir):
            for file in files:
                if file.lower().endswith(video_extensions):
                    full_path = os.path.join(root, file)
                    rel_path = os.path.relpath(full_path, video_dir)

                    # For your normal videos, set label=0
                    videos.append({
                        'video_path': rel_path,
                        'full_path': full_path,
                        'label': 0,  # All are normal videos
                        'class': 'Normal',
                        'filename': file,
                        'directory': os.path.basename(root) if root != video_dir else 'root'
                    })

        print(f"✅ Found {len(videos)} video files in {video_dir}")
        return videos

    @staticmethod
    def create_single_split(videos):
        """Create single split - ALL videos for training"""
        # All videos go to training (no test split)
        splits = {
            'train': videos,  # All 430 videos for training
            'test': []        # Empty test set
        }

        print(f"✅ Created single split for anomaly detection training:")
        print(f"   Training videos: {len(videos)} (ALL videos for training)")
        print(f"   Testing videos: 0 (Will use UCF-Crime test set later)")
        print(f"   Note: These are NORMAL videos only for training normal patterns")

        return splits

    @staticmethod
    def save_metadata(splits, metadata_path):
        """Save metadata to file"""
        metadata = {
            'splits': splits,
            'total_videos': len(splits['train']) + len(splits['test']),
            'train_count': len(splits['train']),
            'test_count': len(splits['test']),
            'created_at': datetime.now().isoformat(),
            'note': 'All videos are NORMAL (label=0) for training normal patterns',
            'dataset_type': 'training_normal_only'
        }

        with open(metadata_path, 'wb') as f:
            pickle.dump(metadata, f)
        print(f"✅ Metadata saved to: {metadata_path}")
        return metadata

    @staticmethod
    def load_metadata(metadata_path):
        """Load metadata from file"""
        if os.path.exists(metadata_path):
            with open(metadata_path, 'rb') as f:
                metadata = pickle.load(f)
            print(f"✅ Metadata loaded from: {metadata_path}")
            return metadata
        return None

    @staticmethod
    def get_video_info(video_path):
        """Get video information using OpenCV"""
        try:
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                return None

            fps = cap.get(cv2.CAP_PROP_FPS)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            duration = total_frames / fps if fps > 0 else 0

            cap.release()

            return {
                'fps': fps,
                'total_frames': total_frames,
                'width': width,
                'height': height,
                'duration': duration,
                'size_mb': os.path.getsize(video_path) / (1024 * 1024)
            }
        except:
            return None

# Load or create metadata
metadata_path = os.path.join(metadata_dir, 'dataset_metadata.pkl')

print("\n" + "=" * 70)
print("LOADING DATASET METADATA")
print("=" * 70)

# Force recreate metadata to fix the split issue
print("⚠️  Force recreating metadata to fix train/test split issue...")

# Delete old metadata file
if os.path.exists(metadata_path):
    print(f"🗑️  Deleting old metadata file: {metadata_path}")
    os.remove(metadata_path)

# Get all videos
all_videos = CustomDatasetMetadata.get_all_videos(DatasetConfig.INPUT_VIDEO_DIR)

if len(all_videos) == 0:
    print(f"❌ No videos found in {DatasetConfig.INPUT_VIDEO_DIR}")
    print("Please check the path and try again.")
    # List what's in the directory
    print(f"\nContents of {DatasetConfig.INPUT_VIDEO_DIR}:")
    if os.path.exists(DatasetConfig.INPUT_VIDEO_DIR):
        for item in os.listdir(DatasetConfig.INPUT_VIDEO_DIR):
            print(f"  - {item}")
else:
    # Create SINGLE split - ALL videos for training
    print(f"\n📊 Creating new metadata with NO train/test split...")
    splits = CustomDatasetMetadata.create_single_split(all_videos)

    # Save metadata
    metadata = CustomDatasetMetadata.save_metadata(splits, metadata_path)

    # Show sample video info
    print(f"\n📊 Sample video information:")
    sample_video = all_videos[0]
    video_info = CustomDatasetMetadata.get_video_info(sample_video['full_path'])
    if video_info:
        print(f"  Video: {sample_video['filename']}")
        print(f"  FPS: {video_info['fps']:.1f}")
        print(f"  Frames: {video_info['total_frames']}")
        print(f"  Resolution: {video_info['width']}x{video_info['height']}")
        print(f"  Duration: {video_info['duration']:.1f} seconds")
        print(f"  Size: {video_info['size_mb']:.1f} MB")

    # Explain the setup
    print(f"\n📚 DATASET SETUP EXPLANATION:")
    print(f"  • You have {len(all_videos)} NORMAL videos from Training-Normal-Videos-Part-1")
    print(f"  • ALL videos will be used for TRAINING normal patterns")
    print(f"  • NO test split (test set will come from UCF-Crime)")
    print(f"  • This is standard for anomaly detection: train on normal only")
    print(f"  • Later, you'll need UCF-Crime anomalous videos")


LOADING DATASET METADATA
⚠️  Force recreating metadata to fix train/test split issue...
🗑️  Deleting old metadata file: /content/drive/MyDrive/Graduation_project/features/metadata/dataset_metadata.pkl
✅ Found 430 video files in /content/drive/MyDrive/Graduation_project/Training-Normal-Videos-Part-1

📊 Creating new metadata with NO train/test split...
✅ Created single split for anomaly detection training:
   Training videos: 430 (ALL videos for training)
   Testing videos: 0 (Will use UCF-Crime test set later)
   Note: These are NORMAL videos only for training normal patterns
✅ Metadata saved to: /content/drive/MyDrive/Graduation_project/features/metadata/dataset_metadata.pkl

📊 Sample video information:
  Video: Normal_Videos152_x264.mp4
  FPS: 30.0
  Frames: 8010
  Resolution: 320x240
  Duration: 267.0 seconds
  Size: 43.9 MB

📚 DATASET SETUP EXPLANATION:
  • You have 430 NORMAL videos from Training-Normal-Videos-Part-1
  • ALL videos will be used for TRAINING normal patterns
  • NO 

In [ ]:
class CustomDatasetMetadata:
    """Load and manage custom dataset metadata"""

    @staticmethod
    def get_all_videos(video_dir):
        """Get all videos from dataset directory"""
        videos = []

        if not os.path.exists(video_dir):
            print(f"❌ Video directory not found: {video_dir}")
            return videos

        # Get all video files
        video_extensions = ('.avi', '.mp4', '.mov', '.mkv', '.flv', '.wmv')

        for root, dirs, files in os.walk(video_dir):
            for file in files:
                if file.lower().endswith(video_extensions):
                    full_path = os.path.join(root, file)
                    rel_path = os.path.relpath(full_path, video_dir)

                    # For your normal videos, set label=0
                    videos.append({
                        'video_path': rel_path,
                        'full_path': full_path,
                        'label': 0,  # All are normal videos
                        'class': 'Normal',
                        'filename': file,
                        'directory': os.path.basename(root) if root != video_dir else 'root'
                    })

        print(f"✅ Found {len(videos)} video files in {video_dir}")
        return videos

    @staticmethod
    def create_single_split(videos):
        """Create single split - ALL videos for training"""
        # All videos go to training (no test split)
        splits = {
            'train': videos,  # All 430 videos for training
            'test': []        # Empty test set
        }

        print(f"✅ Created single split for anomaly detection training:")
        print(f"   Training videos: {len(videos)} (ALL videos for training)")
        print(f"   Testing videos: 0 (Will use UCF-Crime test set later)")
        print(f"   Note: These are NORMAL videos only for training normal patterns")

        return splits

    @staticmethod
    def save_metadata(splits, metadata_path):
        """Save metadata to file"""
        metadata = {
            'splits': splits,
            'total_videos': len(splits['train']) + len(splits['test']),
            'train_count': len(splits['train']),
            'test_count': len(splits['test']),
            'created_at': datetime.now().isoformat(),
            'note': 'All videos are NORMAL (label=0) for training normal patterns'
        }

        with open(metadata_path, 'wb') as f:
            pickle.dump(metadata, f)
        print(f"✅ Metadata saved to: {metadata_path}")
        return metadata

    @staticmethod
    def load_metadata(metadata_path):
        """Load metadata from file"""
        if os.path.exists(metadata_path):
            with open(metadata_path, 'rb') as f:
                metadata = pickle.load(f)
            print(f"✅ Metadata loaded from: {metadata_path}")
            return metadata
        return None

    @staticmethod
    def get_video_info(video_path):
        """Get video information using OpenCV"""
        try:
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                return None

            fps = cap.get(cv2.CAP_PROP_FPS)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            duration = total_frames / fps if fps > 0 else 0

            cap.release()

            return {
                'fps': fps,
                'total_frames': total_frames,
                'width': width,
                'height': height,
                'duration': duration,
                'size_mb': os.path.getsize(video_path) / (1024 * 1024)
            }
        except:
            return None

# Load or create metadata
metadata_path = os.path.join(metadata_dir, 'dataset_metadata.pkl')

print("\n" + "=" * 70)
print("LOADING DATASET METADATA")
print("=" * 70)

# Try loading existing metadata
metadata = CustomDatasetMetadata.load_metadata(metadata_path)

if metadata is None:
    print("No existing metadata found. Scanning video directory...")

    # Get all videos
    all_videos = CustomDatasetMetadata.get_all_videos(DatasetConfig.INPUT_VIDEO_DIR)

    if len(all_videos) == 0:
        print(f"❌ No videos found in {DatasetConfig.INPUT_VIDEO_DIR}")
        print("Please check the path and try again.")
        # List what's in the directory
        print(f"\nContents of {DatasetConfig.INPUT_VIDEO_DIR}:")
        if os.path.exists(DatasetConfig.INPUT_VIDEO_DIR):
            for item in os.listdir(DatasetConfig.INPUT_VIDEO_DIR):
                print(f"  - {item}")
    else:
        # Create SINGLE split - ALL videos for training
        splits = CustomDatasetMetadata.create_single_split(all_videos)

        # Save metadata
        metadata = CustomDatasetMetadata.save_metadata(splits, metadata_path)

        # Show sample video info
        print(f"\n📊 Sample video information:")
        sample_video = all_videos[0]
        video_info = CustomDatasetMetadata.get_video_info(sample_video['full_path'])
        if video_info:
            print(f"  Video: {sample_video['filename']}")
            print(f"  FPS: {video_info['fps']:.1f}")
            print(f"  Frames: {video_info['total_frames']}")
            print(f"  Resolution: {video_info['width']}x{video_info['height']}")
            print(f"  Duration: {video_info['duration']:.1f} seconds")
            print(f"  Size: {video_info['size_mb']:.1f} MB")

        # Explain the setup
        print(f"\n📚 DATASET SETUP EXPLANATION:")
        print(f"  • You have 430 NORMAL videos from Training-Normal-Videos-Part-1")
        print(f"  • ALL videos will be used for TRAINING normal patterns")
        print(f"  • Later, you'll need UCF-Crime anomalous videos for training")
        print(f"  • And UCF-Crime test set for evaluation")
        print(f"  • This is standard for anomaly detection: train on normal only")
else:
    print(f"✅ Using existing metadata")
    splits = metadata['splits']
    print(f"   Total videos: {metadata['total_videos']}")
    print(f"   Training videos: {metadata['train_count']} (normal only)")
    print(f"   Testing videos: {metadata['test_count']} (empty)")
    print(f"   Created at: {metadata['created_at']}")
    if 'note' in metadata:
        print(f"   Note: {metadata['note']}")


LOADING DATASET METADATA
✅ Metadata loaded from: /content/drive/MyDrive/Graduation_project/features/metadata/dataset_metadata.pkl
✅ Using existing metadata
   Total videos: 430
   Training videos: 430 (normal only)
   Testing videos: 0 (empty)
   Created at: 2026-02-03T12:11:08.862186
   Note: All videos are NORMAL (label=0) for training normal patterns


In [ ]:
class VideoPreprocessor:
    """Preprocess video files for feature extraction"""

    def __init__(self, frame_size=(224, 224), max_frames=3000):
        self.frame_size = frame_size
        self.max_frames = max_frames

        # Transformation pipeline
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(frame_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

    def read_video(self, video_path, target_fps=8):
        """
        Read video file and return preprocessed frames at a target FPS.

        Args:
            video_path (str): Path to video file
            target_fps (int): Desired frames per second for output

        Returns:
            frames: List of preprocessed frames as tensors
            fps: Effective FPS after resampling
            video_info: Dictionary with video metadata
        """
        if not os.path.exists(video_path):
            print(f"❌ Video file not found: {video_path}")
            return None, None, None

        try:
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                print(f"❌ Cannot open video: {video_path}")
                return None, None, None

            # Get original video properties
            original_fps = cap.get(cv2.CAP_PROP_FPS) or DatasetConfig.FPS
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            duration = total_frames / original_fps if original_fps > 0 else 0

            # Determine frame sampling interval
            frame_interval = max(1, int(original_fps // target_fps))
            effective_fps = original_fps / frame_interval

            video_info = {
                'fps': effective_fps,
                'original_fps': original_fps,
                'total_frames': total_frames,
                'width': width,
                'height': height,
                'duration': duration,
                'path': video_path
            }

            print(f"🎬 Reading video: {os.path.basename(video_path)}")
            print(f"  Original FPS: {original_fps:.1f}, Target FPS: {effective_fps:.1f}, "
                  f"Frames: {total_frames}, Size: {width}x{height}")

            # Read frames
            frames = []
            frame_count = 0
            read_count = 0

            with tqdm(total=min(total_frames, self.max_frames),
                      desc="Reading frames") as pbar:
                while True:
                    ret, frame = cap.read()
                    if not ret or read_count >= self.max_frames:
                        break

                    # Skip frames to match target FPS
                    if frame_count % frame_interval != 0:
                        frame_count += 1
                        continue

                    # Convert BGR → RGB
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                    # Apply transformation to tensor
                    frame_tensor = self.transform(frame_rgb)
                    frames.append(frame_tensor)

                    frame_count += 1
                    read_count += 1
                    pbar.update(1)

            cap.release()

            if len(frames) == 0:
                print(f"❌ No frames read from: {video_path}")
                return None, None, None

            print(f"✅ Successfully read {len(frames)} frames")
            return frames, effective_fps, video_info

        except Exception as e:
            print(f"❌ Error reading video {video_path}: {e}")
            import traceback
            traceback.print_exc()
            return None, None, None

    def create_segments(self, frames, segment_length=16):
        """
        Create fixed-length segments from frames

        Args:
            frames: List of frame tensors
            segment_length: Number of frames per segment

        Returns:
            segments: List of segments [segment_length, C, H, W]
        """
        if len(frames) < segment_length:
            # Pad with last frame
            padding_needed = segment_length - len(frames)
            frames = frames + [frames[-1]] * padding_needed

        segments = []
        for i in range(0, len(frames), segment_length):
            segment_frames = frames[i:i+segment_length]

            # Pad if necessary
            if len(segment_frames) < segment_length:
                padding_needed = segment_length - len(segment_frames)
                segment_frames.extend([segment_frames[-1]] * padding_needed)

            # Stack frames: [segment_length, C, H, W]
            segment = torch.stack(segment_frames)
            segments.append(segment)

        print(f"📊 Created {len(segments)} segments of {segment_length} frames each")
        return segments

    def save_frames(self, frames, output_dir, video_name, max_save=10):
        """Save sample frames for visualization"""
        os.makedirs(output_dir, exist_ok=True)

        # Save a few sample frames
        save_indices = np.linspace(0, len(frames)-1,
                                  min(max_save, len(frames)), dtype=int)

        for i, idx in enumerate(save_indices):
            # Convert tensor to numpy for saving
            frame_np = frames[idx].permute(1, 2, 0).numpy()
            # Denormalize
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            frame_np = frame_np * std + mean
            frame_np = np.clip(frame_np, 0, 1)
            frame_np = (frame_np * 255).astype(np.uint8)

            save_path = os.path.join(output_dir,
                                   f"{video_name}_frame_{i:03d}.jpg")
            Image.fromarray(frame_np).save(save_path)

        print(f"💾 Saved {len(save_indices)} sample frames to {output_dir}")

# Initialize preprocessor
preprocessor = VideoPreprocessor(
    frame_size=DatasetConfig.FRAME_SIZE,
    max_frames=DatasetConfig.MAX_FRAMES
)

In [ ]:
class YOLOObjectFeatureExtractor:
    """
    Object-centric feature extractor using YOLOv8
    """

    def __init__(self, model_name="yolov8n.pt", device="cuda"):
        from ultralytics import YOLO
        self.model = YOLO(model_name)
        self.model.model.eval()
        self.device = device

        # COCO classes + bbox stats
        self.num_classes = 80
        self.feature_dim = 80 + 6  # counts + (mean + std of w,h,conf)

    def extract_segment_features(self, frames_np):
        """
        frames_np: list of RGB uint8 images [H,W,3]
        """

        results = self.model(frames_np, verbose=False)

        obj_counts = np.zeros(self.num_classes, dtype=np.float32)
        bbox_stats = []

        for res in results:
            for box in res.boxes:
                cls = int(box.cls)
                conf = float(box.conf)
                x1, y1, x2, y2 = box.xyxy[0]

                if cls < self.num_classes:
                    obj_counts[cls] += 1

                bbox_stats.append([
                    (x2 - x1),
                    (y2 - y1),
                    conf
                ])

        # Normalize counts by number of frames
        obj_counts /= max(len(frames_np), 1)

        if bbox_stats:
            bbox_stats = np.array(bbox_stats)
            mean = bbox_stats.mean(axis=0)
            std = bbox_stats.std(axis=0)
            bbox_features = np.concatenate([mean, std])
        else:
            bbox_features = np.zeros(6, dtype=np.float32)

        return np.concatenate([obj_counts, bbox_features])

class YOLOFeatureAdapter:
    """
    Adapter so YOLO can be used inside a multi-stream pipeline
    """

    def __init__(self, yolo_extractor, device="cuda"):
        self.yolo = yolo_extractor
        self.device = device
        self.feature_dim = yolo_extractor.feature_dim

    def extract_features(self, segments):
        """
        segments: list of tensors [T, C, H, W]
        """
        features = []

        for seg in segments:
            frames_np = [
                (f.permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
                for f in seg
            ]
            feat = self.yolo.extract_segment_features(frames_np)
            features.append(feat)

        return np.stack(features)


class BaseFeatureExtractor(nn.Module):
    """Base class for feature extractors"""

    def __init__(self, device='cuda'):
        super().__init__()
        self.device = device
        self.model = None
        self.feature_dim = None

    def forward(self, x):
        """Extract features from input tensor"""
        raise NotImplementedError

    def extract_features(self, segments):
        """
        Extract features from video segments

        Args:
            segments: List of segments [segment_length, C, H, W]

        Returns:
            features: numpy array [num_segments, feature_dim]
        """
        self.eval()
        features = []

        with torch.no_grad():
            for segment in segments:
                # Add batch dimension: [1, segment_length, C, H, W]
                segment_batch = segment.unsqueeze(0).to(self.device)

                # Permute dimensions from [B, T, C, H, W] to [B, C, T, H, W]
                segment_batch = segment_batch.permute(0, 2, 1, 3, 4)

                # Extract features
                feat = self.forward(segment_batch)
                features.append(feat.cpu().numpy().flatten())

        return np.array(features)



class FeatureExtractorFactory:
    """Factory class for different feature extractors"""

    @staticmethod
    def create_extractor(model_type='i3d', device='cuda'):
        """
        Create feature extractor model

        Args:
            model_type: 'c3d', 'i3d', 'r3d', 'lightweight', or 'yolo'
            device: 'cuda' or 'cpu'
        """
        if model_type == 'c3d':
            return C3DFeatureExtractor(device)
        elif model_type == 'i3d':
            return I3DFeatureExtractor(device)
        elif model_type == 'r3d':
            return R3DFeatureExtractor(device)
        elif model_type == 'lightweight':
            return LightweightFeatureExtractor(device)
        else:
            raise ValueError(f"Unknown model type: {model_type}")


class C3DFeatureExtractor(BaseFeatureExtractor):
    """C3D feature extractor using torchvision models"""

    def __init__(self, device='cuda'):
        super().__init__(device)
        self.feature_dim = 4096
        self.model = nn.Sequential(
            # Block 1
            nn.Conv3d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)),

            # Block 2
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv3d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),

            # Block 4
            nn.Conv3d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),

            # Block 5
            nn.Conv3d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),

            # Global pooling
            nn.AdaptiveAvgPool3d((1, 1, 1)),
            nn.Flatten()
        )
          self.model = self.model.to(device)
          self.model.eval()

    def forward(self, x):
        return self.model(x)


from pytorchvideo.models.hub import i3d_r50


class I3DFeatureExtractor(BaseFeatureExtractor):
    """
    I3D Feature Extractor using pretrained I3D-ResNet50
    """

    def __init__(self,
                 device="cuda",
                 pretrained=True,
                 freeze=True):
        super().__init__(device)

        # Load pretrained I3D
        self.model = i3d_r50(pretrained=pretrained)

        # Remove classification head
        self.model.blocks[-1] = nn.Identity()

        # Feature dimension after global pooling
        self.feature_dim = 2048

        if freeze:
            for p in self.model.parameters():
                p.requires_grad = False

        self.model = self.model.to(self.device)
        self.model.eval()

    def forward(self, x):
        """
        Args:
            x: Tensor [B, C, T, H, W]

        Returns:
            features: Tensor [B, feature_dim]
        """

        # Forward through backbone
        feat = self.model(x)  # [B, 2048, 1, 1, 1]

        # Flatten
        feat = feat.view(feat.size(0), -1)

        return feat


class R3DFeatureExtractor(BaseFeatureExtractor):
    """R3D-18 feature extractor"""

    def __init__(self, device='cuda'):
        super().__init__(device)
        self.feature_dim = 512

        try:
            import torchvision.models.video as video_models
            self.model = video_models.r3d_18(pretrained=True)

            # Remove classification head
            self.model = nn.Sequential(*list(self.model.children())[:-1])

            # Add adaptive pooling for consistency
            self.model = nn.Sequential(
                self.model,
                nn.AdaptiveAvgPool3d((1, 1, 1)),
                nn.Flatten()
            )

            # Update feature dimension based on actual model
            with torch.no_grad():
                dummy = torch.randn(1, 3, 16, 112, 112).to(device)
                output = self.model(dummy)
                self.feature_dim = output.shape[1]

            self.model = self.model.to(device)
            self.model.eval()
            print(f"✅ Loaded R3D-18 feature extractor with feature_dim={self.feature_dim}")

        except ImportError as e:
            print(f"⚠️ Could not load R3D-18: {e}, using simplified version")
            self.model = self._build_simple_r3d()
            self.model = self.model.to(device)
            self.model.eval()

    def _build_simple_r3d(self):
        """Build simplified R3D model"""
        model = nn.Sequential(
            # Stem
            nn.Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3)),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),

            # Residual blocks (simplified)
            self._make_layer(64, 64, 2),
            self._make_layer(64, 128, 2, stride=2),
            self._make_layer(128, 256, 2, stride=2),
            self._make_layer(256, 512, 2, stride=2),

            # Pooling
            nn.AdaptiveAvgPool3d((1, 1, 1)),
            nn.Flatten()
        )
        return model

    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        """Create a layer with residual blocks"""
        layers = []

        # First block with downsample if needed
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm3d(out_channels)
            )

        layers.append(ResidualBlock3D(in_channels, out_channels, stride, downsample))

        # Additional blocks
        for _ in range(1, blocks):
            layers.append(ResidualBlock3D(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class ResidualBlock3D(nn.Module):
    """3D Residual Block"""

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3,
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3,
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class LightweightFeatureExtractor(BaseFeatureExtractor):
    """Lightweight feature extractor for Colab"""

    def __init__(self, device='cuda'):
        super().__init__(device)
        self.feature_dim = 512

        self.model = nn.Sequential(
            # Conv3D layers
            nn.Conv3d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm3d(512),
            nn.ReLU(inplace=True),

            # Global pooling
            nn.AdaptiveAvgPool3d((1, 1, 1)),
            nn.Flatten()
        )

        self.model = self.model.to(device)
        self.model.eval()
        print("✅ Loaded lightweight feature extractor")

    def forward(self, x):
        return self.model(x)


class TwoStreamFeatureExtractor:
    """
    Fuses I3D (motion) + YOLO (object) features
    """

    def __init__(self, motion_extractor, object_extractor):
        self.motion_extractor = motion_extractor
        self.object_extractor = object_extractor
        self.feature_dim = (
            motion_extractor.feature_dim +
            object_extractor.feature_dim
        )

    def extract_features(self, segments):
        motion_feats = self.motion_extractor.extract_features(segments)
        object_feats = self.object_extractor.extract_features(segments)

        assert motion_feats.shape[0] == object_feats.shape[0]

        return np.concatenate([motion_feats, object_feats], axis=1)


# Create feature extractor based on configuration
print("\n" + "=" * 70)
print(f"INITIALIZING MOTION EXTRACTOR: {DatasetConfig.MOTION_EXTRACTOR.upper()}")
print("=" * 70)

# Create motion extractor (3D CNN)
motion_extractor = FeatureExtractorFactory.create_extractor(
    model_type=DatasetConfig.MOTION_EXTRACTOR,
    device=device
)

# Create YOLO object extractor
yolo_core = YOLOObjectFeatureExtractor(
    model_name="yolov8n.pt",
    device=device
)

object_extractor = YOLOFeatureAdapter(
    yolo_extractor=yolo_core,
    device=device
)

# Two-stream fusion
feature_extractor = TwoStreamFeatureExtractor(
    motion_extractor,
    object_extractor
)

print(f"✅ Feature extractor initialized")
print(f"   Type: {DatasetConfig.MOTION_EXTRACTOR}")
print(f"   Feature dimension: {feature_extractor.feature_dim}")
print(f"   Device: {device}")


INITIALIZING FEATURE EXTRACTOR: C3D
⚠️ Could not load torchvision model: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same, using simplified C3D
✅ Feature extractor initialized
   Type: c3d
   Feature dimension: 4096
   Device: cuda


In [ ]:
class FeatureExtractionPipeline:
    """Pipeline for extracting and saving features"""

    def __init__(self, input_video_dir, features_dir, metadata_dir, device='cuda'):
        self.input_video_dir = input_video_dir
        self.features_dir = features_dir
        self.metadata_dir = metadata_dir
        self.device = device

        self.preprocessor = VideoPreprocessor(
            frame_size=DatasetConfig.FRAME_SIZE,
            max_frames=DatasetConfig.MAX_FRAMES
        )

        # Main feature extractor
        self.feature_extractor = feature_extractor

        # Progress tracking
        self.progress_file = os.path.join(metadata_dir, 'extraction_progress.json')
        self.progress = self._load_progress()

        # Statistics
        self.stats = {
            'total_videos': 0,
            'successful': 0,
            'failed': 0,
            'total_features': 0,
            'total_size_mb': 0
        }

    # ----------------------------
    # Progress utilities
    # ----------------------------
    def _load_progress(self):
        """Load extraction progress"""
        if os.path.exists(self.progress_file):
            with open(self.progress_file, 'r') as f:
                return json.load(f)
        return {'processed': [], 'failed': [], 'start_time': datetime.now().isoformat()}

    def _save_progress(self):
        """Save extraction progress"""
        with open(self.progress_file, 'w') as f:
            json.dump(self.progress, f, indent=2)

    # ----------------------------
    # Feature saving
    # ----------------------------
    def _save_features(self, features, video_info, split):
        """Save extracted features to NPZ file"""
        video_name = os.path.splitext(video_info['filename'])[0]
        filename = f"{split}_{video_name}.npz"
        filepath = os.path.join(self.features_dir, filename)

        metadata = {
            'video_path': video_info['video_path'],
            'filename': video_info['filename'],
            'label': video_info['label'],
            'class': video_info['class'],
            'split': split,
            'motion_extractor': DatasetConfig.MOTION_EXTRACTOR,
            'feature_dim': features.shape[1],
            'num_segments': features.shape[0],
            'segment_length': DatasetConfig.SEGMENT_LENGTH,
            'extraction_time': datetime.now().isoformat(),
            'video_info': CustomDatasetMetadata.get_video_info(video_info['full_path']),
            'dataset_type': 'training_normal_only'
        }

        np.savez_compressed(
            filepath,
            features=features.astype(np.float32),
            metadata=metadata
        )

        file_size = os.path.getsize(filepath) / (1024 * 1024)  # MB
        self.stats['total_features'] += 1
        self.stats['total_size_mb'] += file_size

        print(f"  ✅ Features saved: {filename}")
        print(f"     Shape: {features.shape[0]} segments × {features.shape[1]} features")
        print(f"     Size: {file_size:.2f} MB")

        return filepath

    # ----------------------------
    # Video processing
    # ----------------------------
    def process_video(self, video_info, split):
        """Process a single video and extract features"""
        video_path = video_info['full_path']

        if not os.path.exists(video_path):
            print(f"❌ Video not found: {video_path}")
            return False

        print(f"\n{'='*60}")
        print(f"Processing: {video_info['filename']}")
        print(f"Split: {split}, Label: {video_info['label']} ({video_info['class']})")
        print(f"Path: {video_path}")
        print(f"{'='*60}")

        try:
            # Step 1: Read video
            frames, fps, video_metadata = self.preprocessor.read_video(video_path)
            if frames is None:
                print(f"❌ Failed to read video")
                return False

            # Step 2: Create segments
            segments = self.preprocessor.create_segments(
                frames,
                segment_length=DatasetConfig.SEGMENT_LENGTH
            )
            if len(segments) == 0:
                print(f"❌ No segments created")
                return False

            # Step 3: Extract features
            print("🔍 Extracting features...")
            start_time = time.time()
            features = self.feature_extractor.extract_features(segments)
            extraction_time = time.time() - start_time

            if features.shape[0] == 0:
                print(f"❌ No features extracted")
                return False

            print(f"✅ Features extracted: {features.shape[0]} segments")
            print(f"⏱️  Extraction time: {extraction_time:.2f} seconds")

            # Step 4: Save features
            self._save_features(features, video_info, split)

            # Step 5: Update progress
            self.progress['processed'].append({
                'filename': video_info['filename'],
                'split': split,
                'time': datetime.now().isoformat(),
                'features_shape': features.shape,
                'extraction_time': extraction_time
            })
            self._save_progress()

            # Step 6: Cleanup
            del frames, segments, features
            torch.cuda.empty_cache()
            gc.collect()

            self.stats['successful'] += 1
            return True

        except Exception as e:
            print(f"❌ Error processing video: {e}")
            import traceback
            traceback.print_exc()

            self.progress['failed'].append({
                'filename': video_info['filename'],
                'split': split,
                'error': str(e),
                'timestamp': datetime.now().isoformat()
            })
            self._save_progress()

            self.stats['failed'] += 1
            return False

    # ----------------------------
    # Extract all features (UPDATED for no test videos)
    # ----------------------------
    def extract_all_features(self, splits, max_videos_per_split=None,
                             resume=True, force_reprocess=False):
        print("=" * 70)
        print("FEATURE EXTRACTION PIPELINE")
        print("=" * 70)

        total_processed = 0

        # Only process training videos
        split = 'train'
        print(f"\n📂 Processing {split.upper()} split (normal videos only)")
        print("-" * 40)

        split_videos = splits[split]
        if max_videos_per_split:
            split_videos = split_videos[:max_videos_per_split]

        processed_count = 0

        for i, video_info in enumerate(tqdm(split_videos, desc=f"Processing {split}")):
            # Check if already processed
            already_processed = any(
                item['filename'] == video_info['filename'] and item['split'] == split
                for item in self.progress['processed']
            )

            if resume and already_processed and not force_reprocess:
                print(f"⏭️  Skipping (already processed): {video_info['filename']}")
                processed_count += 1
                continue

            success = self.process_video(video_info, split)
            if success:
                processed_count += 1

            # Small delay to prevent overwhelming
            time.sleep(0.1)

            # Print progress every 5 videos
            if (i + 1) % 5 == 0:
                print(f"\n📊 Progress: {i+1}/{len(split_videos)} videos")

        total_processed += processed_count
        self.stats['total_videos'] += len(split_videos)

        print(f"\n✅ {split.upper()} split completed:")
        print(f"   Processed: {processed_count}/{len(split_videos)}")

        # Skip test split if empty
        test_videos = splits.get('test', [])
        if len(test_videos) > 0:
            print(f"\n📂 Processing TEST split")
            print("-" * 40)
            print(f"   Test videos: {len(test_videos)}")
            print(f"   Note: Your dataset has no test videos")
        else:
            print(f"\n📂 TEST split: No test videos in dataset")
            print(f"   This is correct for normal training videos")

        print(f"\n{'='*70}")
        print("EXTRACTION SUMMARY")
        print(f"{'='*70}")
        print(f"Total videos: {self.stats['total_videos']}")
        print(f"Successfully processed: {self.stats['successful']}")
        print(f"Failed: {self.stats['failed']}")
        print(f"Feature files created: {self.stats['total_features']}")
        print(f"Total storage used: {self.stats['total_size_mb']:.2f} MB")
        print(f"Progress saved to: {self.progress_file}")

        self._save_progress()
        return self.stats['successful'], self.stats['failed']

    # ----------------------------
    # Analyze features
    # ----------------------------
    def analyze_features(self):
        """Analyze extracted features"""
        feature_files = [f for f in os.listdir(self.features_dir) if f.endswith('.npz')]

        if not feature_files:
            print("❌ No feature files found")
            return

        print(f"\n📊 Found {len(feature_files)} feature files")

        # Collect statistics
        shapes = []
        train_count = 0
        test_count = 0

        for filename in feature_files[:5]:  # Analyze first 5 files
            filepath = os.path.join(self.features_dir, filename)
            data = np.load(filepath, allow_pickle=True)
            features = data['features']
            metadata = data['metadata'].item()

            shapes.append(features.shape)

            if metadata['split'] == 'train':
                train_count += 1
            else:
                test_count += 1

        print(f"\n📈 Feature Analysis:")
        print(f"   Train files: {train_count}")
        print(f"   Test files: {test_count} (should be 0 for normal training)")

        if shapes:
            avg_shape = np.mean([s[0] for s in shapes]), np.mean([s[1] for s in shapes])
            print(f"   Average shape: {avg_shape[0]:.1f} segments × {avg_shape[1]:.1f} features")

            # Show sample
            sample_file = os.path.join(self.features_dir, feature_files[0])
            sample_data = np.load(sample_file, allow_pickle=True)
            sample_features = sample_data['features']
            sample_metadata = sample_data['metadata'].item()

            print(f"\n📋 Sample file: {feature_files[0]}")
            print(f"   Video: {sample_metadata['filename']}")
            print(f"   Label: {sample_metadata['label']} ({sample_metadata['class']})")
            print(f"   Feature shape: {sample_features.shape}")
            print(f"   Extraction time: {sample_metadata['extraction_time']}")

            if 'dataset_type' in sample_metadata:
                print(f"   Dataset type: {sample_metadata['dataset_type']}")


# Initialize pipeline
pipeline = FeatureExtractionPipeline(
    input_video_dir=DatasetConfig.INPUT_VIDEO_DIR,
    features_dir=features_dir,
    metadata_dir=metadata_dir,
    device=device
)

In [ ]:
def process_in_batches(batch_size=50, features_subfolder='normal_one'):
    """
    Process videos in batches to avoid timeouts

    Args:
        batch_size: Number of videos to process in each batch
        features_subfolder: Subfolder to save features in
    """
    print("\n" + "=" * 70)
    print(f"PROCESSING IN BATCHES OF {batch_size} VIDEOS")
    print(f"Saving to: {features_subfolder}")
    print("=" * 70)

    # Create subfolder for features
    normal_one_dir = os.path.join(features_dir, features_subfolder)
    os.makedirs(normal_one_dir, exist_ok=True)
    print(f"✅ Created subfolder: {normal_one_dir}")

    # Load metadata
    metadata_path = os.path.join(metadata_dir, 'dataset_metadata.pkl')
    if not os.path.exists(metadata_path):
        print("❌ Metadata file not found!")
        return

    with open(metadata_path, 'rb') as f:
        metadata = pickle.load(f)

    all_splits = metadata['splits']

    # Initialize pipeline with new features directory
    pipeline = FeatureExtractionPipeline(
        input_video_dir=DatasetConfig.INPUT_VIDEO_DIR,
        features_dir=normal_one_dir,
        metadata_dir=metadata_dir,
        device=device
    )

    total_processed = 0
    total_failed = 0

    # Process TRAINING videos only (no test videos)
    print(f"\n📂 Processing TRAINING split (normal videos only)")
    print("-" * 40)

    train_videos = all_splits['train']
    num_batches = (len(train_videos) + batch_size - 1) // batch_size

    for batch_num in range(num_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(train_videos))
        batch_videos = train_videos[start_idx:end_idx]

        print(f"\n🔄 Batch {batch_num + 1}/{num_batches} "
              f"(videos {start_idx + 1}-{end_idx})")

        # Create split with only training videos
        batch_split = {
            'train': batch_videos,
            'test': []  # Empty test set
        }

        # Process batch
        processed, failed = pipeline.extract_all_features(
            splits=batch_split,
            max_videos_per_split=None,
            resume=True,
            force_reprocess=False
        )

        total_processed += processed
        total_failed += failed

        print(f"\n📊 Batch {batch_num + 1} complete:")
        print(f"   Processed: {processed}")
        print(f"   Failed: {failed}")
        print(f"   Total so far: {total_processed}")

        # Save checkpoint
        checkpoint_file = os.path.join(
            checkpoints_dir,
            f'checkpoint_{features_subfolder}_batch_{batch_num + 1}.json'
        )
        with open(checkpoint_file, 'w') as f:
            json.dump({
                'batch': batch_num + 1,
                'processed': processed,
                'failed': failed,
                'total_processed': total_processed,
                'total_failed': total_failed,
                'timestamp': datetime.now().isoformat(),
                'features_subfolder': features_subfolder,
                'dataset_type': 'normal_training_only'
            }, f, indent=2)

        # Optional: Break between batches to avoid timeout
        if batch_num < num_batches - 1:
            print(f"\n⏳ Waiting 30 seconds before next batch...")
            time.sleep(30)

    print("\n" + "=" * 70)
    print("BATCH PROCESSING COMPLETE")
    print("=" * 70)
    print(f"✅ Total processed: {total_processed}")
    print(f"❌ Total failed: {total_failed}")
    print(f"📁 Features saved to: {normal_one_dir}")
    print(f"\n📝 IMPORTANT NOTES:")
    print(f"   1. All 430 videos are NORMAL (label=0)")
    print(f"   2. These are for TRAINING normal patterns only")
    print(f"   3. You need anomalous videos from UCF-Crime for training")
    print(f"   4. Test set will come from UCF-Crime test videos")

    return total_processed, total_failed

In [ ]:
# Run feature extraction for ALL videos
print("\n" + "=" * 70)
print("PROCESSING NORMAL TRAINING VIDEOS")
print("=" * 70)

print(f"\n📚 DATASET INFORMATION:")
print(f"   • Source: Training-Normal-Videos-Part-1")
print(f"   • Total videos: 430")
print(f"   • Type: NORMAL videos only (label=0)")
print(f"   • Purpose: Training normal patterns for anomaly detection")

# Process all videos in batches of 50
print("\n⚠️ WARNING: This will process ALL 430 NORMAL videos")
print(f"   Batch size: 50 videos per batch")
print(f"   Estimated time: 2-3 hours")
print(f"   Storage needed: ~40-50 MB")

# Ask for confirmation
response = input("\nDo you want to process all NORMAL training videos? (yes/no): ")

if response.lower() == 'yes':
    # Process in batches of 50, save to 'normal_one' subfolder
    processed, failed = process_in_batches(
        batch_size=50,
        features_subfolder='normal_training'
    )

    print(f"\n🎉 NORMAL VIDEO PROCESSING COMPLETED!")
    print(f"   Successfully processed: {processed} videos")
    print(f"   Failed: {failed} videos")

    # Check results
    print("\n" + "=" * 70)
    print("CHECKING RESULTS")
    print("=" * 70)

    # Count feature files in the subfolder
    normal_training_dir = os.path.join(features_dir, 'normal_training')
    if os.path.exists(normal_training_dir):
        feature_files = [f for f in os.listdir(normal_training_dir) if f.endswith('.npz')]
        print(f"✅ Found {len(feature_files)} feature files in {normal_training_dir}")

        # Separate train/test files
        train_files = [f for f in feature_files if f.startswith('train_')]
        test_files = [f for f in feature_files if f.startswith('test_')]

        print(f"   • Training files: {len(train_files)}")
        print(f"   • Testing files: {len(test_files)} (should be 0)")

        # Calculate total size
        total_size = 0
        for f in feature_files:
            try:
                total_size += os.path.getsize(os.path.join(normal_training_dir, f))
            except:
                pass

        print(f"💾 Total storage used: {total_size / (1024*1024):.2f} MB")

        # Show sample files
        if train_files:
            print(f"\n📋 First 5 training feature files:")
            for i, fname in enumerate(train_files[:5]):
                filepath = os.path.join(normal_training_dir, fname)
                file_size = os.path.getsize(filepath) / 1024  # KB

                # Load metadata
                try:
                    with np.load(filepath, allow_pickle=True) as data:
                        metadata = data['metadata'].item()
                    video_name = metadata.get('filename', 'Unknown')
                    segments = metadata.get('num_segments', 'Unknown')
                    label = metadata.get('label', 'Unknown')

                    print(f"  {i+1}. {fname}")
                    print(f"     Video: {video_name}")
                    print(f"     Segments: {segments}")
                    print(f"     Label: {label} (Normal)")
                    print(f"     Size: {file_size:.1f} KB")
                except:
                    print(f"  {i+1}. {fname} ({file_size:.1f} KB)")
    else:
        print(f"❌ Subfolder not found: {normal_training_dir}")

    print(f"\n📚 NEXT STEPS FOR YOUR GRADUATION PROJECT:")
    print(f"   1. ✅ Done: Extracted features from 430 NORMAL training videos")
    print(f"   2. ⏳ Next: Get UCF-Crime anomalous videos (13 categories)")
    print(f"   3. ⏳ Next: Extract features from anomalous videos")
    print(f"   4. ⏳ Next: Train MIL model with normal + anomalous")
    print(f"   5. ⏳ Next: Test on UCF-Crime test set")

else:
    print("\n⏸️  Processing cancelled.")
    print("You can run process_in_batches() later with your preferred batch size.")


PROCESSING NORMAL TRAINING VIDEOS

📚 DATASET INFORMATION:
   • Source: Training-Normal-Videos-Part-1
   • Total videos: 430
   • Type: NORMAL videos only (label=0)
   • Purpose: Training normal patterns for anomaly detection

⚠️ WARNING: This will process ALL 430 NORMAL videos
   Batch size: 50 videos per batch
   Estimated time: 2-3 hours
   Storage needed: ~40-50 MB

PROCESSING IN BATCHES OF 50 VIDEOS
Saving to: normal_training
✅ Created subfolder: /content/drive/MyDrive/Graduation_project/features/normal_training

📂 Processing TRAINING split (normal videos only)
----------------------------------------

🔄 Batch 1/9 (videos 1-50)
FEATURE EXTRACTION PIPELINE

📂 Processing TRAIN split (normal videos only)
----------------------------------------


Processing train:   0%|          | 0/50 [00:00<?, ?it/s]


Processing: Normal_Videos152_x264.mp4
Split: train, Label: 0 (Normal)
Path: /content/drive/MyDrive/Graduation_project/Training-Normal-Videos-Part-1/Normal_Videos152_x264.mp4
🎬 Reading video: Normal_Videos152_x264.mp4
  Original FPS: 30.0, Target FPS: 10.0, Frames: 8010, Size: 320x240



Reading frames:  89%|████████▉ | 2670/3000 [00:09<00:01, 269.72it/s]


✅ Successfully read 2670 frames
📊 Created 167 segments of 16 frames each
🔍 Extracting features...
✅ Features extracted: 167 segments
⏱️  Extraction time: 15.91 seconds
  ✅ Features saved: train_Normal_Videos152_x264.npz
     Shape: 167 segments × 512 features
     Size: 0.24 MB


...


Reading frames:  33%|███▎      | 627/1879 [00:03<00:06, 204.57it/s]


✅ Successfully read 627 frames
📊 Created 40 segments of 16 frames each
🔍 Extracting features...
✅ Features extracted: 40 segments
⏱️  Extraction time: 3.88 seconds
  ✅ Features saved: train_Normal_Videos461_x264.npz
     Shape: 40 segments × 512 features
     Size: 0.06 MB


Processing train: 100%|██████████| 30/30 [09:29<00:00, 18.99s/it]


📊 Progress: 30/30 videos

✅ TRAIN split completed:
   Processed: 30/30

📂 TEST split: No test videos in dataset
   This is correct for normal training videos

EXTRACTION SUMMARY
Total videos: 430
Successfully processed: 430
Failed: 0
Feature files created: 430
Total storage used: 42.65 MB
Progress saved to: /content/drive/MyDrive/Graduation_project/features/metadata/extraction_progress.json

📊 Batch 9 complete:
   Processed: 430
   Failed: 0
   Total so far: 2230

BATCH PROCESSING COMPLETE
✅ Total processed: 2230
❌ Total failed: 0
📁 Features saved to: /content/drive/MyDrive/Graduation_project/features/normal_training

📝 IMPORTANT NOTES:
   1. All 430 videos are NORMAL (label=0)
   2. These are for TRAINING normal patterns only
   3. You need anomalous videos from UCF-Crime for training
   4. Test set will come from UCF-Crime test videos

🎉 NORMAL VIDEO PROCESSING COMPLETED!
   Successfully processed: 2230 videos
   Failed: 0 videos

CHECKING RESULTS
✅ Found 430 feature files in /conte

  1. train_Normal_Videos152_x264.npz
     Video: Normal_Videos152_x264.mp4
     Segments: 167
     Label: 0 (Normal)
     Size: 250.8 KB
  2. train_Normal_Videos004_x264.npz
     Video: Normal_Videos004_x264.mp4
     Segments: 20
     Label: 0 (Normal)
     Size: 31.3 KB
  3. train_Normal_Videos005_x264.npz
     Video: Normal_Videos005_x264.mp4
     Segments: 9
     Label: 0 (Normal)
     Size: 14.4 KB
  4. train_Normal_Videos007_x264.npz
     Video: Normal_Videos007_x264.mp4
     Segments: 21
     Label: 0 (Normal)
     Size: 31.7 KB
  5. train_Normal_Videos008_x264.npz
     Video: Normal_Videos008_x264.mp4
     Segments: 54
     Label: 0 (Normal)
     Size: 79.2 KB

📚 NEXT STEPS FOR YOUR GRADUATION PROJECT:
   1. ✅ Done: Extracted features from 430 NORMAL training videos
   2. ⏳ Next: Get UCF-Crime anomalous videos (13 categories)
   3. ⏳ Next: Extract features from anomalous videos
   4. ⏳ Next: Train MIL model with normal + anomalous
   5. ⏳ Next: Test on UCF-Crime test set


In [ ]:
def check_status(features_subfolder='normal_training'):
    """Check current processing status"""
    print("\n" + "=" * 70)
    print(f"CURRENT PROCESSING STATUS: {features_subfolder}")
    print("=" * 70)

    # Check the subfolder
    subfolder_path = os.path.join(features_dir, features_subfolder)

    if not os.path.exists(subfolder_path):
        print(f"❌ Subfolder not found: {subfolder_path}")
        print(f"   Try: 'normal_one' or 'normal_training'")
        return

    # Count feature files
    feature_files = [f for f in os.listdir(subfolder_path) if f.endswith('.npz')]

    # Count by split
    train_files = [f for f in feature_files if f.startswith('train_')]
    test_files = [f for f in feature_files if f.startswith('test_')]

    print(f"📊 Feature files created: {len(feature_files)}")
    print(f"   • Training files: {len(train_files)} (NORMAL videos)")
    print(f"   • Testing files: {len(test_files)} (should be 0)")

    # Calculate total size
    total_size = 0
    for f in feature_files:
        try:
            total_size += os.path.getsize(os.path.join(subfolder_path, f))
        except Exception as e:
            print(f"   ⚠️ Could not get size for {f}: {e}")

    print(f"💾 Total storage used: {total_size / (1024*1024):.2f} MB")

    # Load metadata to see total video count
    metadata_path = os.path.join(metadata_dir, 'dataset_metadata.pkl')
    if os.path.exists(metadata_path):
        with open(metadata_path, 'rb') as f:
            metadata = pickle.load(f)

        print(f"\n📈 Progress against total dataset:")
        print(f"   • Total normal videos: {metadata['total_videos']}")
        print(f"   • Processed: {len(feature_files)}")
        print(f"   • Remaining: {metadata['total_videos'] - len(feature_files)}")
        print(f"   • Completion: {(len(feature_files) / metadata['total_videos']) * 100:.1f}%")

        if 'note' in metadata:
            print(f"   • Note: {metadata['note']}")

    # Check progress file
    progress_file = os.path.join(metadata_dir, 'extraction_progress.json')
    if os.path.exists(progress_file):
        try:
            with open(progress_file, 'r') as f:
                progress = json.load(f)

            print(f"\n📋 Progress file info:")
            print(f"   • Successfully processed: {len(progress.get('processed', []))}")
            print(f"   • Failed: {len(progress.get('failed', []))}")

            if progress.get('failed'):
                print(f"\n⚠️ Failed videos:")
                for fail in progress['failed'][:5]:  # Show first 5
                    print(f"   • {fail.get('filename', 'Unknown')}")
                if len(progress['failed']) > 5:
                    print(f"   ... and {len(progress['failed']) - 5} more")

        except Exception as e:
            print(f"❌ Could not read progress file: {e}")

    # Check checkpoints
    print(f"\n📁 Checkpoint files:")
    checkpoint_files = [f for f in os.listdir(checkpoints_dir)
                       if f.startswith(f'checkpoint_{features_subfolder}')]

    if checkpoint_files:
        for cf in checkpoint_files[:3]:  # Show first 3
            print(f"   • {cf}")
        if len(checkpoint_files) > 3:
            print(f"   ... and {len(checkpoint_files) - 3} more")
    else:
        print(f"   No checkpoint files found for {features_subfolder}")

    # Summary
    print(f"\n✅ SUMMARY:")
    print(f"   • You are processing NORMAL training videos only")
    print(f"   • These are for learning normal patterns")
    print(f"   • You'll need UCF-Crime anomalous videos next")
    print(f"   • Features saved to: {subfolder_path}")

# Run status check
check_status('normal_training')

print("\n✅ Status check complete!")


CURRENT PROCESSING STATUS: normal_training
📊 Feature files created: 430
   • Training files: 430 (NORMAL videos)
   • Testing files: 0 (should be 0)
💾 Total storage used: 42.65 MB

📈 Progress against total dataset:
   • Total normal videos: 430
   • Processed: 430
   • Remaining: 0
   • Completion: 100.0%
   • Note: All videos are NORMAL (label=0) for training normal patterns

📋 Progress file info:
   • Successfully processed: 430
   • Failed: 0

📁 Checkpoint files:
   • checkpoint_normal_training_batch_1.json
   • checkpoint_normal_training_batch_2.json
   • checkpoint_normal_training_batch_3.json
   ... and 6 more

✅ SUMMARY:
   • You are processing NORMAL training videos only
   • These are for learning normal patterns
   • You'll need UCF-Crime anomalous videos next
   • Features saved to: /content/drive/MyDrive/Graduation_project/features/normal_training

✅ Status check complete!
